# Data Ingestion and Cleaning

We load the reviews and metadata datasets for Health and Personal Care, clean them up and merge them into a single dataframe ready for the next steps.

In [15]:
# Importing libraries
import pandas as pd
import numpy as np
import re
from html import unescape

In [16]:
# Importing the two dataset provided by Deloitte
df_meta = pd.read_json("data/meta_Health_and_Personal_Care.json", lines=True) 
df_review = pd.read_json("data/Health_and_Personal_Care.json", lines=True)

In [17]:
# Analyzing the shape of both dataset
print("Meta shape:", df_meta.shape)
print("Review shape:", df_review.shape)

Meta shape: (60293, 14)
Review shape: (494121, 10)


In [18]:
# Checking the first rows of the two datasets
print("Metadata")
display(df_meta.head())
print("\nReviews")
display(df_review.head())

Metadata


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Health & Personal Care,Silicone Bath Body Brush Exfoliator Shower Bac...,3.9,7,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Rzoeox,[],{'Package Dimensions': '15 x 3.3 x 1.5 inches;...,B07V346GZH,NaN
1,Health & Personal Care,"iPhone 7 Plus 8 Plus Screen Protector, ZHXIN T...",3.8,2,[Tough and Robust: Like all 78X screen protect...,[Features: 2.5D Arc Edge Treatment: The edge i...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ZHXIN,[],"{'Brand': 'ZHXIN', 'Compatible Devices': 'Cell...",B075W927RH,NaN
2,Health & Personal Care,Zig Zag Rolling Machine 70mm Size With FREE BO...,3.9,7,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],None,[],{'Package Dimensions': '4.1 x 1.8 x 0.3 inches...,B01FB26VKY,NaN
3,Health & Personal Care,Sting-Kill Disposable Wipes 8 Each ( Pack of 5),4.1,6,[],"[effective on stings and bites from bees, wasp...",21.37,[{'thumb': 'https://m.media-amazon.com/images/...,[],Sting-kill,[],"{'Brand': 'Sting-kill', 'Item Form': 'Wipe', '...",B01IAI29RU,NaN
4,Health & Personal Care,Heated Eyelash Curler Mini Portable Electric E...,3.3,8,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],BiBOSS,[],{'Package Dimensions': '6.1 x 3.1 x 1.9 inches...,B08CMN38RC,NaN



Reviews


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,12 mg is 12 on the periodic table people! Mg f...,This review is more to clarify someone else’s ...,[],B07TDSJZMR,B07TDSJZMR,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-02-06 00:49:35.902,3,True
1,5,Save the lanet using less plastic.,Love these easy multitasking bleach tablets. B...,[],B08637FWWF,B08637FWWF,AEVWAM3YWN5URJVJIZZ6XPD2MKIA,2020-11-02 22:03:06.880,3,True
2,5,Fantastic,I have been suffering a couple months with hee...,[],B07KJVGNN5,B07KJVGNN5,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,2019-07-24 11:13:58.905,0,True
3,4,It holds the water and makes bubbles. That's ...,"It's cheap and it does what I wanted. The ""ma...",[],B007HY7GC2,B092RP73CX,AEZGPLOYTSAPR3DHZKKXEFPAXUAA,2022-09-04 02:29:02.725,7,True
4,1,Not for me,Didn't do a thing for me. Not saying they don'...,[],B08KYJLF5T,B08KYJLF5T,AEQAYV7RXZEBXMQIQPL6KCT2CFWQ,2022-01-20 23:53:07.262,0,True


In [19]:
# Checking the datatypes
print("Meta dtypes:\n", df_meta.dtypes)
print("\nReview dtypes:\n", df_review.dtypes)

Meta dtypes:
 main_category       object
title               object
average_rating     float64
rating_number        int64
features            object
description         object
price              float64
images              object
videos              object
store               object
categories          object
details             object
parent_asin         object
bought_together    float64
dtype: object

Review dtypes:
 rating                        int64
title                        object
text                         object
images                       object
asin                         object
parent_asin                  object
user_id                      object
timestamp            datetime64[ns]
helpful_vote                  int64
verified_purchase              bool
dtype: object


In [20]:
# Missing values BEFORE converting empty lists/dicts/strings to NaN
meta_missing_raw = df_meta.isna().mean().sort_values(ascending=False)
review_missing_raw = df_review.isna().mean().sort_values(ascending=False)

print("Missing meta (raw):\n", meta_missing_raw.head(15))
print("\nMissing review (raw):\n", review_missing_raw.head(15))

Missing meta (raw):
 bought_together    1.000000
price              0.825253
store              0.038910
main_category      0.000000
title              0.000000
average_rating     0.000000
rating_number      0.000000
features           0.000000
description        0.000000
images             0.000000
videos             0.000000
categories         0.000000
details            0.000000
parent_asin        0.000000
dtype: float64

Missing review (raw):
 rating               0.0
title                0.0
text                 0.0
images               0.0
asin                 0.0
parent_asin          0.0
user_id              0.0
timestamp            0.0
helpful_vote         0.0
verified_purchase    0.0
dtype: float64


## Handling hidden missing values

Some columns have empty lists, empty dicts or empty strings that pandas doesn't see as NaN, so we convert them all.

In [21]:
# Converting [], {}, and "" to NaN in both dataframes
def empty_to_nan(x):
    if x == [] or x == {} or x == "":
        return np.nan
    return x

df_meta   = df_meta.apply(lambda col: col.apply(empty_to_nan))
df_review = df_review.apply(lambda col: col.apply(empty_to_nan))

In [22]:
# Counting missing values AFTER cleanup
meta_missing = df_meta.isna().mean().sort_values(ascending=False)
review_missing = df_review.isna().mean().sort_values(ascending=False)

print("Meta missing ratio (clean):\n", meta_missing.head(20))
print("\nReview missing ratio (clean):\n", review_missing.head(20))

Meta missing ratio (clean):
 categories         1.000000
bought_together    1.000000
videos             0.863384
price              0.825253
features           0.740832
description        0.706765
store              0.038910
details            0.016403
title              0.000083
main_category      0.000000
average_rating     0.000000
rating_number      0.000000
images             0.000000
parent_asin        0.000000
dtype: float64

Review missing ratio (clean):
 images               0.956972
text                 0.000055
rating               0.000000
title                0.000000
asin                 0.000000
parent_asin          0.000000
user_id              0.000000
timestamp            0.000000
helpful_vote         0.000000
verified_purchase    0.000000
dtype: float64


## Filtering

We only keep verified purchases and products with more than 10 reviews to avoid spam and statistically meaningless data.

In [23]:
# Keeping only verified purchases
print(f"Reviews before verified filter: {len(df_review)}")
df_review = df_review[df_review["verified_purchase"] == True].copy()
print(f"Reviews after verified filter:  {len(df_review)}")

Reviews before verified filter: 494121
Reviews after verified filter:  445072


In [24]:
# Keeping only products with more than 10 reviews
MIN_REVIEWS = 10

review_counts = df_review.groupby("parent_asin").size()
valid_asins = review_counts[review_counts > MIN_REVIEWS].index

print(f"Unique products before filter: {df_review['parent_asin'].nunique()}")
df_review = df_review[df_review["parent_asin"].isin(valid_asins)].copy()
print(f"Unique products after filter (>{MIN_REVIEWS} reviews): {df_review['parent_asin'].nunique()}")
print(f"Total reviews kept: {len(df_review)}")

Unique products before filter: 56306
Unique products after filter (>10 reviews): 6971
Total reviews kept: 318435


## Text Cleaning

We remove HTML tags, special characters and extra whitespace from the review title and text.

In [25]:
def clean_text(text):
    """Remove HTML tags, unescape HTML entities, strip special chars and extra whitespace."""
    if pd.isna(text):
        return text
    text = unescape(str(text))        
    text = re.sub(r"<[^>]+>", " ", text)  
    text = re.sub(r"[^\w\s.,!?'-]", " ", text) 
    text = re.sub(r"\s+", " ", text).strip()  
    return text

# Apply to review columns
df_review["title"] = df_review["title"].apply(clean_text)
df_review["text"]  = df_review["text"].apply(clean_text)

df_review[["title", "text"]].head()

,title,text
0,12 mg is 12 on the periodic table people! Mg f...,This review is more to clarify someone else s ...
2,Fantastic,I have been suffering a couple months with hee...
3,It holds the water and makes bubbles. That's w...,It's cheap and it does what I wanted. The mass...
4,Not for me,Didn't do a thing for me. Not saying they don'...
17,returned,it was a nice tray smaller than i expected but...


## Merge metadata and reviews

We pull title, brand and description from the metadata and join them with the reviews on parent_asin.

In [26]:
# Extracting brand from the 'details' dict and flatten description list
def extract_brand(details):
    """Pull 'Brand' from the details dictionary."""
    if isinstance(details, dict):
        return details.get("Brand", np.nan)
    return np.nan

def flatten_description(desc):
    """Join description list into a single string."""
    if isinstance(desc, list) and len(desc) > 0:
        return " ".join(str(d) for d in desc)
    if isinstance(desc, str) and desc:
        return desc
    return np.nan

df_meta["brand"] = df_meta["details"].apply(extract_brand)
df_meta["description_clean"] = df_meta["description"].apply(flatten_description)

# Cleaning text in metadata fields too
df_meta["title"] = df_meta["title"].apply(clean_text)
df_meta["description_clean"] = df_meta["description_clean"].apply(clean_text)
df_meta["brand"] = df_meta["brand"].apply(clean_text)

print(f"Brand coverage: {df_meta['brand'].notna().mean():.2%}")
print(f"Description coverage: {df_meta['description_clean'].notna().mean():.2%}")

Brand coverage: 49.72%
Description coverage: 29.32%


In [27]:
# Selecting relevant metadata columns and merge with reviews
meta_cols = df_meta[["parent_asin", "title", "brand", "description_clean",
    "average_rating", "rating_number", "price", "store"]].copy()

# Renaming to avoid column name clashes after merge
meta_cols.rename(columns={
    "title": "product_title",
    "average_rating": "product_avg_rating",
    "rating_number": "product_rating_count"
}, inplace=True)

# Merging on parent_asin (left join: keep all filtered reviews, attach metadata)
df = df_review.merge(meta_cols, on="parent_asin", how="left")

print(f"Merged dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Merged dataset shape: (318435, 17)
Columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'product_title', 'brand', 'description_clean', 'product_avg_rating', 'product_rating_count', 'price', 'store']


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,brand,description_clean,product_avg_rating,product_rating_count,price,store
0,4,12 mg is 12 on the periodic table people! Mg f...,This review is more to clarify someone else s ...,NaN,B07TDSJZMR,B07TDSJZMR,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-02-06 00:49:35.902,3,True,High Potency Magnesium Citrate Capsules 1000mg...,Life Nutrition,NaN,4.5,470,NaN,Life Nutrition
1,5,Fantastic,I have been suffering a couple months with hee...,NaN,B07KJVGNN5,B07KJVGNN5,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,2019-07-24 11:13:58.905,0,True,"Dr. Foot's Gel Heel Protectors, Plantar Fascii...",NaN,NaN,3.6,78,NaN,Dr.Foot
2,4,It holds the water and makes bubbles. That's w...,It's cheap and it does what I wanted. The mass...,NaN,B007HY7GC2,B092RP73CX,AEZGPLOYTSAPR3DHZKKXEFPAXUAA,2022-09-04 02:29:02.725,7,True,"Homedics Bubble Bliss Deluxe-Foot Spa, Heat Ma...",Homedics,NaN,4.4,8312,NaN,Homedics
3,1,Not for me,Didn't do a thing for me. Not saying they don'...,NaN,B08KYJLF5T,B08KYJLF5T,AEQAYV7RXZEBXMQIQPL6KCT2CFWQ,2022-01-20 23:53:07.262,0,True,Brain Supplement 1053mg - Premium Nootropic Br...,Nature's Nutrition,NaN,4.1,94,NaN,Nature's Nutrition
4,5,returned,it was a nice tray smaller than i expected but...,NaN,B01H0SVP9O,B01H0SVP9O,AHGAOIZVODNHYMNCBV4DECZH42UQ,2020-11-11 14:54:54.192,0,True,North American Walker Tray with Non-Slip Grip Mat,NaN,Walker tray with grip mat. Turn any walker int...,4.0,2072,26.9,North American Health + Wellness


## Daniele1 — Product-level dataset (1 row per product)

### Cosa facciamo
Dopo il **merge** tra `df_review` e `meta_cols`, il dataframe `df` è ancora a livello **review**: lo stesso `parent_asin` compare molte volte (una riga per recensione).
In questa cella costruiamo invece un dataset **a livello prodotto** (`products_df`), con **una sola riga per `parent_asin`**.

### Perché serve (collegamento alla pipeline)
Questo passaggio è necessario per lo **Step 2: Semantic Representation**:
- vogliamo calcolare **un embedding per prodotto**, quindi ci serve un “documento” stabile e coerente per ogni prodotto.
- usare tutte le recensioni insieme sarebbe rumoroso e troppo lungo; qui partiamo da **metadata** (titolo, brand, descrizione) che descrivono l’identità del prodotto.

### Come lo facciamo (logica)
1. **Selezioniamo le colonne metadata rilevanti** (se esistono in `df`), per evitare errori se qualche colonna manca.
2. **Rimuoviamo i duplicati** su `parent_asin` → otteniamo 1 riga per prodotto.
3. Creiamo `product_text_base`: una stringa composta da:
   - `TITLE`
   - `BRAND`
   - `DESCRIPTION`
   Questo campo sarà il testo di input per gli embedding del prodotto nello step successivo.

### Output
- `products_df`: tabella prodotto-level (1 riga per `parent_asin`)
- `product_text_base`: testo pronto per calcolare gli embedding (Step 2)

In [32]:
# --- Daniele: Product-level table (1 row per parent_asin) ---

# metadati unici per prodotto (dopo merge)
meta_unique_cols = [
    "parent_asin", "product_title", "brand", "description_clean",
    "product_avg_rating", "product_rating_count", "price", "store"
]
meta_unique_cols = [c for c in meta_unique_cols if c in df.columns]

products_df = df[meta_unique_cols].drop_duplicates("parent_asin").copy()

# testo base per embedding prodotti (Step 2)
def build_product_text(row):
    parts = []
    if "product_title" in row and pd.notna(row["product_title"]):
        parts.append(f"TITLE: {row['product_title']}")
    if "brand" in row and pd.notna(row["brand"]):
        parts.append(f"BRAND: {row['brand']}")
    if "description_clean" in row and pd.notna(row["description_clean"]):
        parts.append(f"DESCRIPTION: {row['description_clean']}")
    return "\n".join(parts)

products_df["product_text_base"] = products_df.apply(build_product_text, axis=1)

print("Products:", products_df.shape)
products_df[["parent_asin", "product_text_base"]].head(3)

Products: (6971, 9)


,parent_asin,product_text_base
0,B07TDSJZMR,TITLE: High Potency Magnesium Citrate Capsules...
1,B07KJVGNN5,"TITLE: Dr. Foot's Gel Heel Protectors, Plantar..."
2,B092RP73CX,"TITLE: Homedics Bubble Bliss Deluxe-Foot Spa, ..."


## Step Daniele2 — Review-level dataset selezionato (Top-N reviews “utili” per prodotto)

### Cosa facciamo
Anche dopo il merge, `df` contiene **tutte le righe a livello recensione**.  
In questa cella estraiamo un sottoinsieme di recensioni **più informative** per ogni prodotto (`parent_asin`), creando `reviews_top_df`.

### Perché serve (collegamento alla pipeline)
Questo dataset serve a due step della pipeline:

- **Step 2 — Semantic Representation (Review Embeddings):** calcoliamo gli embedding **solo** sulle recensioni più utili, evitando di processare milioni di righe inutili (problema di volume/tempo).
- **Step 3 — Review Intelligence & Summarization:** usiamo questo subset per generare insight e riassunti “pro/contro” basati su recensioni ad alta qualità.

Inoltre, mantenendo le recensioni separate (non concatenate in un unico testo), **non perdiamo la granularità**: possiamo mostrare in demo una “evidence review” che giustifica un risultato.

### Come lo facciamo (logica)
1. **Selezioniamo solo le colonne necessarie** (`review_cols`) e verifichiamo che esistano davvero in `df` (così la cella è robusta).
2. Gestiamo valori mancanti:
   - `helpful_vote` → sostituiamo `NaN` con `0`
   - `text` → riempiamo `NaN`, convertiamo a stringa, facciamo `strip()` e rimuoviamo testi vuoti
3. **Ordiniamo le recensioni per “utilità”**, usando come priorità:
   - `helpful_vote` decrescente (recensioni giudicate utili dagli utenti)
   - poi `rating` decrescente (se disponibile)
4. Per ogni `parent_asin` prendiamo le **prime `TOP_N` recensioni** (`groupby + head`).

### Parametro chiave
- `TOP_N = 10`: controlla quante recensioni tenere per prodotto.
  - valori più bassi → più veloce e più “pulito”
  - valori più alti → più copertura ma più rumore/tempo

### Output
- `reviews_top_df`: recensioni selezionate (Top-N per prodotto), pronte per:
  - embedding review-level (Step 2)
  - summarization e sentiment/keyword extraction (Step 3)

In [33]:
# --- Daniele2: Select top-N useful reviews per product ---

TOP_N = 20  # puoi cambiare a 5/15
review_cols = ["parent_asin", "title", "text", "rating", "verified_purchase", "helpful_vote", "timestamp"]
review_cols = [c for c in review_cols if c in df.columns]

reviews_df = df[review_cols].copy()

# helpful_vote può essere NaN
if "helpful_vote" in reviews_df.columns:
    reviews_df["helpful_vote"] = reviews_df["helpful_vote"].fillna(0)

# testo review già pulito: usiamo 'text'
reviews_df["text"] = reviews_df["text"].fillna("").astype(str).str.strip()
reviews_df = reviews_df[reviews_df["text"].str.len() > 0]

# ranking: helpful_vote desc, poi rating desc (se esiste)
sort_cols = [c for c in ["helpful_vote", "rating"] if c in reviews_df.columns]
reviews_df = reviews_df.sort_values(sort_cols, ascending=[False]*len(sort_cols))

reviews_top_df = reviews_df.groupby("parent_asin", as_index=False).head(TOP_N).copy()

print("Top reviews:", reviews_top_df.shape)
reviews_top_df.head(3)

Top reviews: (120406, 7)


,parent_asin,title,text,rating,verified_purchase,helpful_vote,timestamp
146704,B08X9LB1WC,Germ Guardian creating formaldehyde?!,I'm afraid this product is breaking down mold ...,2,True,5907,2016-02-16 18:49:43.000
184897,B07FTK5DWF,DS or SS? We have both.,"First, we bought the dual speed machine. After...",5,True,1343,2012-08-29 20:47:11.000
109705,B07V25DLDG,This Has Proven to be Quite Effective for Us,My wife and I are in our seventies. We live in...,4,True,1179,2019-12-07 12:23:56.534


In [34]:
counts = reviews_top_df.groupby("parent_asin").size()
print("min reviews per product:", counts.min())
print("max reviews per product:", counts.max())
print("products with < TOP_N reviews:", (counts < TOP_N).sum())

min reviews per product: 10
max reviews per product: 20
products with < TOP_N reviews: 3225


In [30]:
# Saving the cleaned and merged dataset for the next steps
df.to_csv("data/reviews_cleaned.csv", index=False)
products_df.to_csv("data/products_cleaned.csv", index=False)
reviews_top_df.to_csv("data/reviews_topN.csv", index=False)

print("Saved: data/products_cleaned.csv")
print("Saved: data/reviews_topN.csv")

Saved: data/products_cleaned.csv
Saved: data/reviews_topN.csv
